<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/02_dpo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — DPO preference tuning in Colab

This notebook is a small hands-on experiment with **Direct Preference Optimization (DPO)**.

Instead of giving one target answer for each prompt, we provide a `chosen` answer and a `rejected` answer. DPO then trains the model to prefer the chosen responses.

**Important:** this is a teaching experiment. It is DPO with a tiny synthetic preference dataset, not a full production RLHF pipeline. The next step after this notebook is a true SFT → DPO comparison.

In [ ]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.21,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"

In [ ]:
import torch
import transformers, datasets, peft, trl

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before running this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

## 1. Preference data

Each example contains the same prompt plus two candidate responses. `chosen` is the response we want the model to prefer; `rejected` is the response we want it to prefer less.

The examples intentionally emphasize concise, factual biomedical explanations. They are synthetic teaching data, not expert-curated clinical advice.

In [ ]:
from datasets import Dataset

preferences = [
    {
        'prompt': 'Explain what DNA is.',
        'chosen': 'DNA is the molecule that stores hereditary genetic information in cells.',
        'rejected': 'DNA is a thing in biology that is related to genes.'
    },
    {
        'prompt': 'Explain what RNA is.',
        'chosen': 'RNA is a nucleic acid involved in gene expression and several other cellular processes.',
        'rejected': 'RNA is basically DNA but different.'
    },
    {
        'prompt': 'What is a gene?',
        'chosen': 'A gene is a DNA sequence that contributes to a functional product such as an RNA or protein.',
        'rejected': 'A gene is just a piece of DNA that causes a trait.'
    },
    {
        'prompt': 'What is TP53?',
        'chosen': 'TP53 encodes p53, a tumor-suppressor protein that helps regulate cell-cycle responses to cellular stress and DNA damage.',
        'rejected': 'TP53 is a cancer gene that causes tumors.'
    },
    {
        'prompt': 'What is BRCA1?',
        'chosen': 'BRCA1 encodes a protein involved in maintaining genome stability, including roles in DNA damage response and repair.',
        'rejected': 'BRCA1 is simply a gene that gives people breast cancer.'
    },
    {
        'prompt': 'What is a protein?',
        'chosen': 'A protein is a polymer of amino acids that folds into structures enabling specific cellular functions.',
        'rejected': 'A protein is a type of molecule found in food and muscles.'
    },
    {
        'prompt': 'What is a mutation?',
        'chosen': 'A mutation is a change in a DNA sequence; its biological effect depends on its type, location, and context.',
        'rejected': 'A mutation is always harmful.'
    },
    {
        'prompt': 'What is gene expression?',
        'chosen': 'Gene expression is the process by which information encoded in a gene is used to produce a functional RNA or protein.',
        'rejected': 'Gene expression means a gene is turned on and makes DNA.'
    },
]

# Repeat the small preference set to make the learning signal easy to observe.
preferences = preferences * 12
dataset = Dataset.from_list(preferences)
dataset

In [ ]:
# Keep a small held-out set for a qualitative before/after check.
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']
print('Train rows:', len(train_dataset))
print('Eval rows:', len(eval_dataset))

## 2. Load the base model

We use the same small model as Notebook 01 so the experiments are easy to compare. DPO can also use a larger model later; the small model is purely for the Colab demo.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
)
model = model.cuda()

## 3. Baseline generation

First, inspect the model before preference tuning. The point is not to score it rigorously; we want a concrete before/after reference.

In [ ]:
def generate(model, question, max_new_tokens=80):
    prompt = f'### Question:\n{question}\n\n### Answer:\n'
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_QUESTION = 'Why is BRCA1 biologically important?'
print('BEFORE DPO')
print(generate(model, TEST_QUESTION))

## 4. Attach a LoRA adapter and run DPO

This uses PEFT + DPOTrainer. DPO directly optimizes the relative preference between `chosen` and `rejected` responses. The base model stays frozen and only the LoRA adapter is trained.

This notebook starts directly from the base model for simplicity. A more realistic pipeline is **SFT → DPO**, where the DPO stage starts from the SFT model produced by Notebook 01.

In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

dpo_args = DPOConfig(
    output_dir='./outputs/dpo-biobot',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=1e-5,
    beta=0.1,
    max_length=256,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = DPOTrainer(
    model=model,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

trainer.train()

In [ ]:
model.eval()
print('AFTER DPO')
print(generate(model, TEST_QUESTION))

## 5. Inspect the preference signal

Try the following questions before and after DPO:

- Why is BRCA1 biologically important?
- What does TP53 do?
- What is a mutation?
- What is gene expression?

You should think of DPO here as learning a **preference pattern** (concise, qualified, biologically grounded answers), not as adding authoritative biomedical knowledge. The tiny synthetic dataset is far too small for real biomedical knowledge acquisition.

In [ ]:
ADAPTER_DIR = './outputs/dpo-biobot-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved:', ADAPTER_DIR)

## What to learn from this experiment

**LoRA/SFT:** show the model examples of desired outputs.

**DPO:** show the model which of two candidate outputs is preferred.

The next useful experiment is a controlled comparison: **Base → SFT/LoRA → DPO**, using the same evaluation prompts. After that, we can look at reward models and PPO-style RLHF.